This notebook must be executed inside the FINN container provided by Xilinx.

Follow the steps of the [FINN docks](https://finn.readthedocs.io/en/latest/), section [Quickstart](https://finn.readthedocs.io/en/latest/getting_started.html#running-finn-in-docker), to download, build and verify the container installation.

You must also move your project folder to inside the same folder the repository is located.

To start the container, go to the folder where the repo was installed and run $ ./run-docker.sh notebook

If you are using vscode, you can select the notebook kernel inside the container to run the code.

This notebook is strongly based on Xilinx [tfc_end2end_example.ipynb](https://github.com/Xilinx/finn/blob/main/notebooks/end2end_example/bnn-pynq/tfc_end2end_example.ipynb) notebook and 0BAB1 [2_finn_hardware_layers.ipynb](https://github.com/0BAB1/tutorial-snippets/blob/main/8%20Python%20to%20FPGA/2_finn_hardware_layers.ipynb) notebook.

Check them out for deeper instructions.

# Setup Python Paths for Libraries

In [1]:
import os
import sys

# Correct the path where the environment starts to be the same folder of the notebook, so that the imports work correctly.
# When starting the container with the notebook server, the script hardcodes the Jupyter server to start inside the ./notebooks folder.
os.chdir('../QFast-SCNN_with_Brevitas_and_FINN/finn_environment')

# Add the train_environment directory to the system path to allow imports from there
sys.path.append(os.path.abspath('../train_environment'))
print(sys.path)

['/usr/lib/python310.zip', '/usr/lib/python3.10', '/usr/lib/python3.10/lib-dynload', '', '/tmp/home_dir/.local/lib/python3.10/site-packages', '/home/jose-vitor/finn-repo/deps/qonnx/src', '/home/jose-vitor/finn-repo/deps/finn-experimental/src', '/home/jose-vitor/finn-repo/deps/brevitas/src', '/home/jose-vitor/finn-repo/deps/pyverilator', '/home/jose-vitor/finn-repo/src', '/usr/local/lib/python3.10/dist-packages', '/workspace/src/dataset-loading', '/usr/lib/python3/dist-packages', '/home/jose-vitor/finn-repo/QFast-SCNN_with_Brevitas_and_FINN/train_environment']


In [2]:
# Import all used libraries an functions and create all folders needed for the project, for convenience when is needed to run only a part of the project.
import torch
import torch.nn.functional as F
from torchvision import datasets, transforms
from torchvision.datasets import Cityscapes
from torch.utils.data import DataLoader
from pathlib import Path
import numpy as np
from config import IM_SIZE, NUM_CLASSES, DATA_PATH, BIT_WIDTH
from finn.util.visualization import showSrc, showInNetron
from qonnx.core.modelwrapper import ModelWrapper
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames, RemoveStaticGraphInputs
from qonnx.transformation.fold_constants import FoldConstants
from finn.util.pytorch import ToTensor
from qonnx.transformation.merge_onnx_models import MergeONNXModels
from qonnx.core.datatype import DataType
from qonnx.util.cleanup import cleanup as qonnx_cleanup
from onnx import helper
from my_finn_utils import load_state_dict, generate_cityscapes_labels, IdToTrainIdTransform
import models.QFastSCNN as qfscnn
import qonnx.core.onnx_exec as oxe
import onnx.numpy_helper as nph
from finn.transformation.streamline import Streamline
from finn.transformation.streamline.reorder import MoveScalarLinearPastInvariants
import finn.transformation.streamline.absorb as absorb
from finn.transformation.streamline.round_thresholds import RoundAndClipThresholds
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from qonnx.transformation.general import RemoveUnusedTensors
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from qonnx.transformation.general import RemoveUnusedTensors
from finn.transformation.streamline.round_thresholds import RoundAndClipThresholds
from finn.transformation.streamline.reorder import MoveMulPastFork, MoveScalarMulPastConv, MoveScalarMulPastConvTranspose, MoveMulPastDWConv, MoveIdenticalOpPastJoinOp, MoveLinearPastEltwiseAdd, MoveScalarLinearPastInvariants, MoveAddPastFork
from finn.transformation.streamline.absorb import AbsorbMulIntoMultiThreshold, AbsorbAddIntoMultiThreshold
from finn.util.basic import pynq_part_map

# Setup Path
onnx_path = f'../onnx/quant_model_{BIT_WIDTH}_bits.onnx'
onnx_name = Path(onnx_path).stem

brevitas_onnx_path = f'./01_from_brevitas_onnx/{onnx_name}_from_brevitas.onnx'
Path(brevitas_onnx_path).parent.mkdir(parents=True, exist_ok=True)

tidy_path = f'./02_tidy_onnx/{onnx_name}_tidy.onnx'
Path(tidy_path).parent.mkdir(parents=True, exist_ok=True)

preproc_path = f'./03_preproc_onnx/{onnx_name}_preproc.onnx'
Path(preproc_path).parent.mkdir(parents=True, exist_ok=True)

postproc_path = f'./04_postproc_onnx/{onnx_name}_postproc.onnx'
Path(postproc_path).parent.mkdir(parents=True, exist_ok=True)

streamlined_path = f'./05_streamlined_onnx/{onnx_name}_streamlined.onnx'
Path(streamlined_path).parent.mkdir(parents=True, exist_ok=True)

ready_for_hw_path = f'./06_ready_for_hw_conversion_onnx/{onnx_name}_ready_for_hw_conversion.onnx'
Path(ready_for_hw_path).parent.mkdir(parents=True, exist_ok=True)

hw_layers_path = f'./07_with_hw_layers/{onnx_name}_hw.onnx'
Path(hw_layers_path).parent.mkdir(parents=True, exist_ok=True)

df_part_path = f'./08_df_part/{onnx_name}_df_part.onnx'
Path(df_part_path).parent.mkdir(parents=True, exist_ok=True)
df_part_isolated_path = f'./08_df_part/{onnx_name}_df_part_isolated.onnx'
Path(df_part_isolated_path).parent.mkdir(parents=True, exist_ok=True)

fpga_hls_path = f'./09_fpga_hls/{onnx_name}_fpga_hls.onnx'
Path(fpga_hls_path).parent.mkdir(parents=True, exist_ok=True)

folded_model_path = f'./10_folded_model/{onnx_name}_folded_model.onnx'
Path(folded_model_path).parent.mkdir(parents=True, exist_ok=True)

# change this if you have a different PYNQ board, see list above
pynq_board = "Pynq-Z2"
fpga_part = pynq_part_map[pynq_board]
target_clk_ns = 10  # 100 MHz. This is the standard clock frequency used in the FINN tutorial.
TARGET_FPS = 30
target_clk_cycles_per_frame = 1/(TARGET_FPS * 10e-9)

/home/jose-vitor/finn-repo/deps/brevitas/src/brevitas/__init__.py:10: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import DistributionNotFound
/usr/local/lib/python3.10/dist-packages/pkg_resources/__init__.py:2871: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)
/usr/local/lib/python3.10/dist-packages/pkg_resources/__init__.py:2871: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('mpl_toolkits')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packag

# Convert Brevitas exported QONNX to FINN

ModelWrapper is a wrapper around the ONNX model which provides several helper functions to make it easier to work with the model.

ConvertQONNXtoFINN Convert the model to the FINN-ONNX model. The main difference of this ONNX from the standard is how the quantization is handled.

In [ ]:
from finn.util.visualization import showSrc, showInNetron
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.util.cleanup import cleanup as qonnx_cleanup
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from finn.transformation.qonnx.infer_quant_avg_pool_2d import AvgPoolAndTruncToQuantAvgPool
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.infer_datatypes import InferDataTypes
from pathlib import Path
from config import BIT_WIDTH

# Setup Path
onnx_path = f'../onnx/quant_model_{BIT_WIDTH}_bits.onnx'
onnx_name = Path(onnx_path).stem

# Note: The original tfc_end2end_example.ipynb does not run the InferShapes and InferDataTypes transformations, but without them the ConvertQONNXtoFINN transformation will fail with an error of missing shape information.
qonnx_cleanup(onnx_path, out_file=onnx_path)
model = ModelWrapper(onnx_path)
model = model.transform(InferShapes())
model = model.transform(InferDataTypes())
model = model.transform(ConvertQONNXtoFINN())

brevitas_onnx_path = f'./01_from_brevitas_onnx/{onnx_name}_from_brevitas.onnx'
Path(brevitas_onnx_path).parent.mkdir(parents=True, exist_ok=True)
model.save(brevitas_onnx_path)

/home/jose-vitor/finn-repo/deps/qonnx/src/qonnx/util/onnx.py:40: DeprecationWarning: `mapping.TENSOR_TYPE_TO_NP_TYPE` is now deprecated and will be removed in a future release.To silence this warning, please use `helper.tensor_dtype_to_np_dtype` instead.
  return np.zeros(dims, dtype=onnx.mapping.TENSOR_TYPE_TO_NP_TYPE[vi.type.tensor_type.elem_type])


In [4]:
showInNetron(brevitas_onnx_path)

Serving './from_brevitas_onnx/quant_model_8_bits_from_brevitas.onnx' at http://0.0.0.0:8081


# Setup Model for FINN

* Tidy up (and also after EACH step)
* Pre (data feed) / Post proc (top k)
* Model streamlining (Main step) + smaller example
* Model HW Layers (Generates Matrix Vector Activation Units for fc layers)
* Model data flow partitions (Generate a sub-graph for all HW convertible nodes)
* Specialize layer, ready for hw conversion (generates hls for the dataflow partition node)

### Tidy Up

In [ ]:
from finn.util.visualization import showSrc, showInNetron
from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames, RemoveStaticGraphInputs
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.core.modelwrapper import ModelWrapper
from config import IM_SIZE

model = ModelWrapper(brevitas_onnx_path)

# TIDY UP
model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(InferDataTypes())
model = model.transform(RemoveStaticGraphInputs())
tidy_path = f'./02_tidy_onnx/{onnx_name}_tidy.onnx'
Path(tidy_path).parent.mkdir(parents=True, exist_ok=True)
model.save(tidy_path)

### Pre processing

FINN model expects UINT8 input. According to Xilinx, this is highly beneficial for performance, because you can directly input raw data to the model, instead of relying on CPU for pre processing.

The the QFast-SCNN model exported to QONNX has the pre processing layers integrated in the Pytorch model, so the expected input is already from 0 to 255.

If the target model was trained with tensor inputs different than [0, 255], like the standard torch.Tensor [0, 1] or tensors with Imagenet normalization, you have two main options to follow:
* Modify your Pytorch model only for the QONNX export, integrating the pre processing inside the model (the option I have chosen for QFast-SCNN).
* Add the preprocessing layers in the QONNX model, following the "Adding Pre- and Postprocessing" section of Xilinx [tfc_end2end_example.ipynb](https://github.com/Xilinx/finn/blob/main/notebooks/end2end_example/bnn-pynq/tfc_end2end_example.ipynb) notebook.

In [3]:
from finn.util.pytorch import ToTensor
from qonnx.transformation.merge_onnx_models import MergeONNXModels
from qonnx.core.datatype import DataType
from qonnx.util.cleanup import cleanup as qonnx_cleanup
import torch
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from onnx import helper

# PRE PROC : NONE
model = ModelWrapper(tidy_path)

# add input annotation: UINT8 is what we will feed the model during inference
global_inp_name = model.graph.input[0].name
model.set_tensor_datatype(global_inp_name, DataType["UINT8"])

# verify if the sizes of first and last node outputs are correct
first_node_out = model.graph.node[0].output[0]
last_node_out = model.graph.node[-1].output[0]
print(f"\nOutput shape shape of first node ({model.graph.node[0].op_type}): {model.get_tensor_shape(first_node_out)}")
print(f"Output shape shape of last node ({model.graph.node[-1].op_type}): {model.get_tensor_shape(last_node_out)}")

# verify if input and output datatypes are correct
print(f"\nInput datatype: {model.get_tensor_datatype(global_inp_name)}")
print(f"Output datatype: {model.get_tensor_datatype(last_node_out)}")

# Save the preprocessed model
preproc_path = f'./03_preproc_onnx/{onnx_name}_preproc.onnx'
Path(preproc_path).parent.mkdir(parents=True, exist_ok=True)
model.save(preproc_path)

# Print a human readable representation of the graph
#print(helper.printable_graph(model.graph))


Output shape shape of first node (Mul): [1, 3, 1024, 1024]
Output shape shape of last node (Add): [1, 19, 128, 128]

Input datatype: UINT8
Output datatype: FLOAT32


In [10]:
showInNetron(preproc_path)

Serving './03_preproc_onnx/quant_model_8_bits_preproc.onnx' at http://0.0.0.0:8081


### Compare the Outputs of Pytorch Model and QONNX Model

Optional but highly recommended step. The Pytorch model will be loaded with the "finn" mode so the test input of this model is the same as the QONNX model.

In [4]:
import torch
from torchvision import transforms
from torchvision.datasets import Cityscapes
from my_finn_utils import load_state_dict, generate_cityscapes_labels, IdToTrainIdTransform
import models.QFastSCNN as qfscnn
from config import NUM_CLASSES, DATA_PATH, BIT_WIDTH, IM_HEIGHT

lable_conversion, id_names = generate_cityscapes_labels()

# Defining the Cityscapes validation dataset.
val_dataset = Cityscapes(
    root=DATA_PATH,
    split='val',
    mode='fine',
    target_type='semantic',
    transform=transforms.Compose([
        transforms.PILToTensor(), # Converting the PIL images to tensors, keeping the original pixel values (0-255) which is important for the quantized model that expects UINT8 inputs.
        transforms.CenterCrop(IM_HEIGHT) # center crop of 1024x1024
    ]),
    target_transform=transforms.Compose([
        transforms.PILToTensor(), # Converting the PIL masks to tensors, keeping the original pixel values (0-255).
        transforms.CenterCrop(IM_HEIGHT), # center crop of 1024x1024
        IdToTrainIdTransform(lable_conversion), # Converting the original Cityscapes labels to the 19 classes used for training and evaluation, as per the Cityscapes benchmark.
    ])
)

# Importing the test image and mask
img_tensor, smnt_tensor = val_dataset[0]
img_tensor = img_tensor.unsqueeze(0) # Add batch dimension
print("Input image shape:", img_tensor.shape)
print("Input image dtype:", img_tensor.dtype)
print("Input mask shape:", smnt_tensor.shape)
print("Input mask dtype:", smnt_tensor.dtype)

# Creating a Brevitas model instance and loading the quantized weights from the training phase.
brevitas_model = qfscnn.QFastSCNN(NUM_CLASSES, mode="finn")
brevitas_model = load_state_dict(brevitas_model, path=f"../train_environment/model_weights/quant_params/best_{BIT_WIDTH}_bit_quant_model.pth", strict=False)
brevitas_model.eval();

Input image shape: torch.Size([1, 3, 1024, 1024])
Input image dtype: torch.uint8
Input mask shape: torch.Size([1, 1024, 1024])
Input mask dtype: torch.uint8
Carregando modelo best_8_bit_quant_model


In [5]:
import torch.nn.functional as F

# Run a foward pass on Brevitas model
with torch.inference_mode():
    brevitas_output = brevitas_model(img_tensor)
brevitas_output_upsampled = F.interpolate(brevitas_output, size=img_tensor.shape[2:], mode='bilinear', align_corners=False)
brevitas_output_mask = torch.softmax(brevitas_output_upsampled, dim=1).argmax(dim=1).to(torch.uint8)

# Calculate how many pixels are matching between the Brevitas output mask and the ground truth mask, and print the results.
matching_pixels = (brevitas_output_mask == smnt_tensor).sum().item()

print(f"Output shape from Brevitas model: {brevitas_output.shape}\n"
      f"Output shape after upsampling: {brevitas_output_upsampled.shape}\n"
      f"Output mask shape: {brevitas_output_mask.shape}\n"
      f"Accuracy: {(100 * matching_pixels/smnt_tensor.numel()):.2f}%\n")

/usr/local/lib/python3.10/dist-packages/torch/_tensor.py:1255: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at ../c10/core/TensorImpl.h:1758.)
  return super(Tensor, self).rename(names)
/usr/local/lib/python3.10/dist-packages/torch/overrides.py:1528: DeprecationWarning: Defining your `__torch_function__ as a plain method is deprecated and will be an error in future, please define it as a classmethod.
  warnings.warn("Defining your `__torch_function__ as a plain method is deprecated and "
/home/jose-vitor/finn-repo/QFast-SCNN_with_Brevitas_and_FINN/train_environment/models/QFastSCNN.py:71: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  output = torch.c

Output shape from Brevitas model: torch.Size([1, 19, 128, 128])
Output shape after upsampling: torch.Size([1, 19, 1024, 1024])
Output mask shape: torch.Size([1, 1024, 1024])
Accuracy: 81.17%



In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
from qonnx.core.modelwrapper import ModelWrapper
import qonnx.core.onnx_exec as oxe
import onnx.numpy_helper as nph

model = ModelWrapper(preproc_path)
print(f"\nInput datatype: {model.get_tensor_datatype(global_inp_name)}")

# add input annotation: UINT8 is what we will feed the model during inference
global_inp_name = model.graph.input[0].name
model.set_tensor_datatype(global_inp_name, DataType["UINT8"])

### ----------------------------------------------------------------------------------------------------------- ###
### --- BEGIN OF MONKEY PATCH TO AVOID THE ERROR OF MISSING SHAPES ON THE RESIZE NODES WITH EMPTY ROI INPUT --- ###
### ----------------------------------------------------------------------------------------------------------- ###

# 1. Make sure that the ignored wires of Resize are strictly empty
for n in model.graph.node:
    if n.op_type in ["Resize", "Upsample"] and len(n.input) > 2:
        n.input[1] = ""

# 2.GLOBAL PATCH: modification of the make ModelWrapper.check_all_tensor_shapes_specified to make the model's shape checking ignore empty strings.
def patched_check(self, fix_missing_init_shape=False):
    graph = self._model_proto.graph
    ret = True
    for n in graph.node:
        for i in n.input:
            if i != "":  # Ignore empty strings when checking for tensor shapes
                ret = (self.get_tensor_shape(i, fix_missing_init_shape=fix_missing_init_shape) is not None) and ret
        for o in n.output:
            if o != "":
                ret = (self.get_tensor_shape(o, fix_missing_init_shape=fix_missing_init_shape) is not None) and ret
    return ret

ModelWrapper.check_all_tensor_shapes_specified = patched_check

# 3. Apply the transformations to infer shapes and fold constants.
model = model.transform(InferShapes())
model = model.transform(FoldConstants())

# 4. Set the initializer for the empty string to avoid errors when executing the model.
model.set_initializer("", np.empty([0], dtype=np.float32))

### --------------------------------------------------------------------------------------------------------- ###
### --- END OF MONKEY PATCH TO AVOID THE ERROR OF MISSING SHAPES ON THE RESIZE NODES WITH EMPTY ROI INPUT --- ###
### --------------------------------------------------------------------------------------------------------- ###

# Run the QONNX inference.
input_tensor = img_tensor.detach().numpy().astype(np.float32) # ONNX runtime expects the input tensor to be of type float32, so we convert it to that type before passing it to the model.
input_dict = {"global_in": input_tensor}
output_dict = oxe.execute_onnx(model, input_dict)
qonnx_output = output_dict[list(output_dict.keys())[0]]

# Upsampling the QONNX output to compare to the ground truth mask from the Brevitas model.
qonnx_output_upsampled = F.interpolate(torch.from_numpy(qonnx_output), size=img_tensor.shape[2:], mode='bilinear', align_corners=False)
qonnx_output_mask = torch.softmax(qonnx_output_upsampled, dim=1).argmax(dim=1).to(torch.uint8)

# Calculate how many pixels are matching between the Brevitas output mask and the ground truth mask, and print the results.
matching_pixels = (qonnx_output_mask == smnt_tensor).sum().item()

print(f"Output shape from QONNX model: {qonnx_output.shape}\n"
      f"Output shape after upsampling: {qonnx_output_upsampled.shape}\n"
      f"Output mask shape: {qonnx_output_mask.shape}\n"
      f"Accuracy: {(100 * matching_pixels/smnt_tensor.numel()):.2f}%\n")


Input datatype: UINT8


/home/jose-vitor/finn-repo/deps/qonnx/src/qonnx/util/basic.py:296: UserWarning: The values of tensor MultiThreshold_57_out0 can't be represented with the set datatype annotation (INT8), they will be rounded to match the datatype annotation.
  warnings.warn(
/home/jose-vitor/finn-repo/deps/qonnx/src/qonnx/util/basic.py:296: UserWarning: The values of tensor MultiThreshold_58_out0 can't be represented with the set datatype annotation (INT8), they will be rounded to match the datatype annotation.
  warnings.warn(
/home/jose-vitor/finn-repo/deps/qonnx/src/qonnx/util/basic.py:296: UserWarning: The values of tensor MultiThreshold_59_out0 can't be represented with the set datatype annotation (INT8), they will be rounded to match the datatype annotation.
  warnings.warn(
/home/jose-vitor/finn-repo/deps/qonnx/src/qonnx/util/basic.py:296: UserWarning: The values of tensor MultiThreshold_60_out0 can't be represented with the set datatype annotation (INT8), they will be rounded to match the dataty

Output shape from QONNX model: (1, 19, 128, 128)
Output shape after upsampling: torch.Size([1, 19, 1024, 1024])
Output mask shape: torch.Size([1, 1024, 1024])
Accuracy: 81.20%



In [11]:
# check output types
print(f"Brevitas output dtype: {brevitas_output.dtype}\nQONNX output dtype: {qonnx_output.dtype}\n")

# check if outputs are close enough
matching_pixels = (brevitas_output_mask == qonnx_output_mask).sum().item()
total_pixels = brevitas_output_mask.numel()
print(f"Matching pixels: {matching_pixels}/{total_pixels} ({(100 * matching_pixels/total_pixels):.2f}%)\n")

Brevitas output dtype: torch.float32
QONNX output dtype: float32

Matching pixels: 1044739/1048576 (99.63%)



### Post-Processing

In [ ]:
# POST PROC
model = ModelWrapper(preproc_path)

# tidy-up again
model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(InferDataTypes())
model = model.transform(RemoveStaticGraphInputs())

# Save the postprocessed model
postproc_path = f'./04_postproc_onnx/{onnx_name}_postproc.onnx'
Path(postproc_path).parent.mkdir(parents=True, exist_ok=True)
model.save(postproc_path)

### Model streamlining

Streamlining transformations listed in the bellow cell has the goal to eliminate floating point operations by moving them around and collapsing them in the previous step transforming them into multi-thresholding nodes.

In [8]:
from finn.transformation.streamline import Streamline
showSrc(Streamline)

class Streamline(Transformation):
    """Apply the streamlining transform, see arXiv:1709.04060."""

    def apply(self, model):
        streamline_transformations = [
            ConvertSubToAdd(),
            ConvertDivToMul(),
            BatchNormToAffine(),
            ConvertSignToThres(),
            MoveMulPastMaxPool(),
            MoveScalarLinearPastInvariants(),
            AbsorbSignBiasIntoMultiThreshold(),
            MoveAddPastMul(),
            MoveScalarAddPastMatMul(),
            MoveAddPastConv(),
            MoveScalarMulPastMatMul(),
            MoveScalarMulPastConv(),
            MoveAddPastMul(),
            CollapseRepeatedAdd(),
            CollapseRepeatedMul(),
            MoveMulPastMaxPool(),
            AbsorbAddIntoMultiThreshold(),
            FactorOutMulSignMagnitude(),
            AbsorbMulIntoMultiThreshold(),
            Absorb1BitMulIntoMatMul(),
            Absorb1BitMulIntoConv(),
            RoundAndClipThresholds(),
        ]
        for tr

In [ ]:
from finn.transformation.streamline import Streamline
# we can see the list of apllied transformations here : showSrc(Streamline)
from finn.transformation.streamline.reorder import MoveScalarLinearPastInvariants
import finn.transformation.streamline.absorb as absorb

model = ModelWrapper(postproc_path)

# STREAMLINE
model = model.transform(Streamline())

# Save the streamlined model
streamlined_path = f'./05_streamlined_onnx/{onnx_name}_streamlined.onnx'
Path(streamlined_path).parent.mkdir(parents=True, exist_ok=True)
model.save(streamlined_path)

# Print a human readable representation of the graph
#print(helper.printable_graph(model.graph))

In [12]:
showInNetron(streamlined_path)

Serving './streamlined_onnx/quant_model_8_bits_streamlined.onnx' at http://0.0.0.0:8081


After the streamlining process you may notice some Mul and Add blocks included in your model graph on Netron. This blocks are terrible for hardware conversion since they will be converted to float32 operations.

Those blocks must be absorved in MultiThreshold blocks so that they be handled by quantization scale factor and zero point. The FINN streamline stardard transformations, which can be checked one of the previous notebook cells, do not include every use case of all ML models, so the user must look up the [FINN standard model transformations](https://finn.readthedocs.io/en/latest/source_code/finn.transformation.streamline.html#module-finn.transformation.streamline) and use the ones suitable for their model.

Regarding this project, I have used several standard transforms, bus also had to create two new ones, since AveragePool and Concat are blocks not included in the FINN transforms.

In [ ]:
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from qonnx.transformation.general import RemoveUnusedTensors
from finn.transformation.streamline.round_thresholds import RoundAndClipThresholds
from finn.transformation.streamline.reorder import MoveMulPastFork, MoveScalarMulPastConv, MoveScalarMulPastConvTranspose, MoveMulPastDWConv, MoveIdenticalOpPastJoinOp, MoveLinearPastEltwiseAdd, MoveScalarLinearPastInvariants, MoveAddPastFork
from finn.transformation.streamline.absorb import AbsorbMulIntoMultiThreshold, AbsorbAddIntoMultiThreshold
from finn.transformation.streamline.collapse_repeated import CollapseRepeatedAdd, CollapseRepeatedMul, CollapseRepeatedOp

import custom_finn_transformations as cft

model = ModelWrapper(streamlined_path)

# running other transformations to clean up the graph as much as possible, to make it ready for the hardware conversion
model = model.transform(InferDataLayouts())
model = model.transform(RemoveUnusedTensors())
model = model.transform(MoveMulPastFork())
model = model.transform(MoveScalarMulPastConvTranspose())
model = model.transform(MoveScalarMulPastConv())
model = model.transform(MoveMulPastDWConv())
model = model.transform(MoveLinearPastEltwiseAdd())
model = model.transform(MoveAddPastFork())
model = model.transform(MoveScalarLinearPastInvariants())
#model = model.transform(cft.MoveMulPastAvgPool())
model = model.transform(cft.MoveScalarLinearPastConcat())
model = model.transform(AbsorbMulIntoMultiThreshold())
model = model.transform(AbsorbAddIntoMultiThreshold())
model = model.transform(CollapseRepeatedAdd())
model = model.transform(CollapseRepeatedMul())
model = model.transform(RoundAndClipThresholds())

# Cleaning up the graph and inferring shapes again, to make it ready for the hardware conversion.
model = model.transform(RemoveUnusedTensors())
model = model.transform(InferShapes())

# Save the streamlined model
ready_for_hw_path = f'./06_ready_for_hw_conversion_onnx/{onnx_name}_ready_for_hw_conversion.onnx'
Path(ready_for_hw_path).parent.mkdir(parents=True, exist_ok=True)
model.save(ready_for_hw_path)

# Print a human readable representation of the graph
print(helper.printable_graph(model.graph))

/home/jose-vitor/finn-repo/deps/qonnx/src/qonnx/transformation/infer_data_layouts.py:127: UserWarning: Assuming 4D input is NCHW
  warnings.warn("Assuming 4D input is NCHW")


graph main_graph (
  %global_in[FLOAT, 1x3x1024x1024]
) initializers (
  %Resize_3_param0[FLOAT, 4]
  %Resize_2_param0[FLOAT, 4]
  %Resize_1_param0[FLOAT, 4]
  %Resize_0_param0[FLOAT, 4]
  %Resize_4_param0[FLOAT, 4]
  %Add_7_param0[FLOAT, 1x19x1x1]
  %Conv_4_param0[FLOAT, 64x48x1x1]
  %Conv_8_param0[FLOAT, 64x384x1x1]
  %Conv_26_param0[FLOAT, 128x576x1x1]
  %Conv_33_param0[FLOAT, 128x1x32x32]
  %Conv_40_param0[FLOAT, 32x128x1x1]
  %Conv_34_param0[FLOAT, 128x1x16x16]
  %Conv_38_param0[FLOAT, 32x128x1x1]
  %Conv_35_param0[FLOAT, 128x1x8x8]
  %Conv_39_param0[FLOAT, 32x128x1x1]
  %Conv_36_param0[FLOAT, 128x1x4x4]
  %Conv_37_param0[FLOAT, 32x128x1x1]
  %Conv_42_param0[FLOAT, 128x1x3x3]
  %Conv_44_param0[FLOAT, 128x1x3x3]
  %Conv_45_param0[FLOAT, 128x128x1x1]
  %Conv_48_param0[FLOAT, 19x128x1x1]
  %Mul_27_param0[FLOAT, scalar]
  %Conv_0_param0[FLOAT, 32x3x3x3]
  %Conv_1_param0[FLOAT, 32x1x3x3]
  %Conv_2_param0[FLOAT, 48x32x1x1]
  %Conv_3_param0[FLOAT, 48x1x3x3]
  %Conv_5_param0[FLOAT, 384x64

In [9]:
showInNetron(ready_for_hw_path)

Serving './06_ready_for_hw_conversion_onnx/quant_model_8_bits_ready_for_hw_conversion.onnx' at http://0.0.0.0:8081


### Compare the Outputs of Ready for HW Convertion ONNX Model with Outputs of Pre-Proc QONNX Model

Another optional step but highly recommended if the model has unusual layers to FINN framework or if custom QONNX transforms were added to the streamline process (which is the case for my implementation of QFast-SCNN)

In [10]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
from qonnx.core.modelwrapper import ModelWrapper
import qonnx.core.onnx_exec as oxe
import onnx.numpy_helper as nph

# add input annotation: UINT8 is what we will feed the model during inference
global_inp_name = model.graph.input[0].name
model.set_tensor_datatype(global_inp_name, DataType["UINT8"])

model = ModelWrapper(ready_for_hw_path)
print(f"\nInput datatype: {model.get_tensor_datatype(global_inp_name)}")

### ----------------------------------------------------------------------------------------------------------- ###
### --- BEGIN OF MONKEY PATCH TO AVOID THE ERROR OF MISSING SHAPES ON THE RESIZE NODES WITH EMPTY ROI INPUT --- ###
### ----------------------------------------------------------------------------------------------------------- ###

# 1. Make sure that the ignored wires of Resize are strictly empty
for n in model.graph.node:
    if n.op_type in ["Resize", "Upsample"] and len(n.input) > 2:
        n.input[1] = ""

# 2.GLOBAL PATCH: modification of the make ModelWrapper.check_all_tensor_shapes_specified to make the model's shape checking ignore empty strings.
def patched_check(self, fix_missing_init_shape=False):
    graph = self._model_proto.graph
    ret = True
    for n in graph.node:
        for i in n.input:
            if i != "":  # Ignora a string vazia!
                ret = (self.get_tensor_shape(i, fix_missing_init_shape=fix_missing_init_shape) is not None) and ret
        for o in n.output:
            if o != "":
                ret = (self.get_tensor_shape(o, fix_missing_init_shape=fix_missing_init_shape) is not None) and ret
    return ret

ModelWrapper.check_all_tensor_shapes_specified = patched_check

# 3. Apply the transformations to infer shapes and fold constants.
model = model.transform(InferShapes())
model = model.transform(FoldConstants())

# 4. Set the initializer for the empty string to avoid errors when executing the model.
model.set_initializer("", np.empty([0], dtype=np.float32))

### --------------------------------------------------------------------------------------------------------- ###
### --- END OF MONKEY PATCH TO AVOID THE ERROR OF MISSING SHAPES ON THE RESIZE NODES WITH EMPTY ROI INPUT --- ###
### --------------------------------------------------------------------------------------------------------- ###

# Run the QONNX inference.
input_tensor = img_tensor.detach().numpy().astype(np.float32) # ONNX runtime expects the input tensor to be of type float32, so we convert it to that type before passing it to the model.
input_dict = {"global_in": input_tensor}
output_dict = oxe.execute_onnx(model, input_dict)
hw_qonnx_output = output_dict[list(output_dict.keys())[0]]

# Upsampling the QONNX output to compare to the ground truth mask from the Brevitas model.
hw_qonnx_output_upsampled = F.interpolate(torch.from_numpy(hw_qonnx_output), size=img_tensor.shape[2:], mode='bilinear', align_corners=False)
hw_qonnx_output_mask = torch.softmax(hw_qonnx_output_upsampled, dim=1).argmax(dim=1).to(torch.uint8)

# Calculate how many pixels are matching between the Brevitas output mask and the ground truth mask, and print the results.
matching_pixels = (hw_qonnx_output_mask == smnt_tensor).sum().item()

print(f"Output shape from QONNX model: {hw_qonnx_output.shape}\n"
      f"Output shape after upsampling: {hw_qonnx_output_upsampled.shape}\n"
      f"Output mask shape: {hw_qonnx_output_mask.shape}\n"
      f"Accuracy: {(100 * matching_pixels/smnt_tensor.numel()):.2f}%\n")


Input datatype: UINT8


/home/jose-vitor/finn-repo/deps/qonnx/src/qonnx/util/onnx.py:40: DeprecationWarning: `mapping.TENSOR_TYPE_TO_NP_TYPE` is now deprecated and will be removed in a future release.To silence this warning, please use `helper.tensor_dtype_to_np_dtype` instead.
  return np.zeros(dims, dtype=onnx.mapping.TENSOR_TYPE_TO_NP_TYPE[vi.type.tensor_type.elem_type])
/home/jose-vitor/finn-repo/deps/qonnx/src/qonnx/util/basic.py:296: UserWarning: The values of tensor MultiThreshold_0_param0 can't be represented with the set datatype annotation (UINT8), they will be rounded to match the datatype annotation.
  warnings.warn(
/home/jose-vitor/finn-repo/deps/qonnx/src/qonnx/util/basic.py:296: UserWarning: The values of tensor MultiThreshold_14_param0 can't be represented with the set datatype annotation (INT8), they will be rounded to match the datatype annotation.
  warnings.warn(
/home/jose-vitor/finn-repo/deps/qonnx/src/qonnx/util/basic.py:296: UserWarning: The values of tensor MultiThreshold_20_param0 c

Output shape from QONNX model: (1, 19, 128, 128)
Output shape after upsampling: torch.Size([1, 19, 1024, 1024])
Output mask shape: torch.Size([1, 1024, 1024])
Accuracy: 81.20%



In [12]:
# check output types
print(f"Brevitas output dtype: {brevitas_output.dtype}\nQONNX output dtype: {qonnx_output.dtype}\n")

# check if outputs are close enough
matching_pixels = (hw_qonnx_output_mask == qonnx_output_mask).sum().item()
total_pixels = hw_qonnx_output_mask.numel()
print(f"Matching pixels: {matching_pixels}/{total_pixels} ({(100 * matching_pixels/total_pixels):.2f}%)\n")

Brevitas output dtype: torch.float32
QONNX output dtype: float32

Matching pixels: 1045020/1048576 (99.66%)



## Convert to HW

This step does not generate HLS nor RTL code, but rather merges thresholds and matrix vector operations (or convolutions if you use conv nets) into Matrix Vector Activation Units (or Sliding Window Units), if possible, that each represent a layer that FINN can easily work with. Note that an input preprocessing quantiazer will be implemented as a simple standalone Threasholding layer that will then nbe converted to Thresholding_hls layer.

In [43]:
import finn.transformation.fpgadataflow.convert_to_hw_layers as to_hw
from qonnx.core.modelwrapper import ModelWrapper
from finn.transformation.streamline.round_thresholds import RoundAndClipThresholds
from finn.transformation.streamline.absorb import AbsorbConsecutiveTransposes, AbsorbTransposeIntoMultiThreshold, AbsorbTransposeIntoResize, AbsorbConsecutiveTransposes
from finn.transformation.streamline.reorder import MakeScaleResizeNHWC, MoveTransposePastJoinAdd
from qonnx.transformation.lower_convs_to_matmul import LowerConvsToMatMul
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.infer_datatypes import InferDataTypes
import custom_finn_transformations as cft

# TO HW LAYERS

model = ModelWrapper(ready_for_hw_path)
model = model.transform(LowerConvsToMatMul())
model = model.transform(to_hw.InferConvInpGen())
model = model.transform(to_hw.InferAddStreamsLayer())
model = model.transform(to_hw.InferChannelwiseLinearLayer())
model = model.transform(to_hw.InferPool())
model = model.transform(to_hw.InferQuantizedMatrixVectorActivation())
model = model.transform(to_hw.InferVectorVectorActivation())
model = model.transform(RoundAndClipThresholds())
model = model.transform(AbsorbTransposeIntoMultiThreshold())
model = model.transform(to_hw.InferThresholdingLayer())

# Adjustments to the model to make it compatible with the hardware conversion, including reordering and absorbing operations.
model = model.transform(InferShapes())
model = model.transform(InferDataLayouts())
model = model.transform(AbsorbTransposeIntoResize())
#model = model.transform(MakeScaleResizeNHWC())
model = model.transform(to_hw.InferUpsample())
model = model.transform(MoveTransposePastJoinAdd())
model = model.transform(AbsorbConsecutiveTransposes())
model = model.transform(AbsorbTransposeIntoResize())
model = model.transform(AbsorbTransposeIntoMultiThreshold())

# Runs the cuustom transformation to transform Concat nodes from NCHW to NHWC, and them convert them to hardware compatible layers.
model = model.transform(cft.MakeConcatNHWC())
model = model.transform(InferShapes())
model = model.transform(InferDataTypes())
model = model.transform(to_hw.InferConcatLayer())

# Insert InferDuplicateStreamsLayer to deal with skip connections in the model, which are not supported by the hardware conversion.
model = model.transform(to_hw.InferDuplicateStreamsLayer())
model = model.transform(RemoveUnusedTensors())

# Save the hw model
hw_layers_path = f'./07_with_hw_layers/{onnx_name}_hw.onnx'
Path(hw_layers_path).parent.mkdir(parents=True, exist_ok=True)
model.save(hw_layers_path)

#print(helper.printable_graph(model.graph))

/home/jose-vitor/finn-repo/src/finn/transformation/fpgadataflow/convert_to_hw_layers.py:668: UserWarning: Broadcasting Mul(Mul_27)
  warnings.warn("Broadcasting " + str(node.op_type) + "(" + node.name + ")")
/home/jose-vitor/finn-repo/src/finn/custom_op/fpgadataflow/fmpadding.py:130: UserWarning: inputDataType changing for FMPadding_Batch_: UINT8 -> FLOAT32 
  warnings.warn(warn_str)
/home/jose-vitor/finn-repo/src/finn/custom_op/fpgadataflow/vectorvectoractivation.py:182: UserWarning: inputDataType changing for VVAU_: UINT8 -> FLOAT32 
  warnings.warn(warn_str)
/home/jose-vitor/finn-repo/src/finn/custom_op/fpgadataflow/fmpadding.py:130: UserWarning: inputDataType changing for FMPadding_Batch_: FLOAT32 -> UINT8 
  warnings.warn(warn_str)
/home/jose-vitor/finn-repo/src/finn/custom_op/fpgadataflow/vectorvectoractivation.py:182: UserWarning: inputDataType changing for VVAU_: FLOAT32 -> UINT8 
  warnings.warn(warn_str)


In [45]:
showInNetron(hw_layers_path)

Serving './07_with_hw_layers/quant_model_8_bits_hw.onnx' at http://0.0.0.0:8081


### Isolate HW convertible layers.

Once MVAU are infered, we can ask FINN to discriminate HW convertible layers into dataflow partitions.

In [46]:
from finn.transformation.fpgadataflow.create_dataflow_partition import CreateDataflowPartition

model = ModelWrapper(hw_layers_path)
parent_model = model.transform(CreateDataflowPartition())

# Save the dataflow partition model
df_part_path = f'./08_df_part/{onnx_name}_df_part.onnx'
Path(df_part_path).parent.mkdir(parents=True, exist_ok=True)
parent_model.save(df_part_path)

In [48]:
showInNetron(df_part_path)

Serving './08_df_part/quant_model_8_bits_df_part.onnx' at http://0.0.0.0:8081


We select the child streaming dataflow model to work on it (i.e. do the actual HLS/RTL conversion)

In [21]:
from qonnx.custom_op.registry import getCustomOp
parent_model = ModelWrapper(df_part_path)
sdp_node = parent_model.get_nodes_by_op_type("StreamingDataflowPartition")[0]
sdp_node = getCustomOp(sdp_node)
dataflow_model_filename = sdp_node.get_nodeattr("model")

### MVAU HLS Conversion

With the entire model isolated with only HW abstraction layers we can replace them by actual HLS or RTL module using FINN. This is done by calling the Specializelayers Transform. It is also possible before that to let the FINN flow now if the preference of implementation is HLS layers or RTL layers by setting the nodeattr "preferred_impl_style" as "hls" or "rtl". If this attr is left empty, than by default the preferred implementation will be HLS. If RTL and FINN does not have the implementation of this node on RTL, then it will be set back to HLS.

For this project the target FPGA is Pynq-Z2 with RTL layers. The "preferred_impl_style" noteattr preferred type is RTL, since the goal of this project get the most optimized model on FPGA.

In [51]:
# print the names of the supported PYNQ boards
from finn.util.basic import pynq_part_map
print(pynq_part_map.keys())

# change this if you have a different PYNQ board, see list above
pynq_board = "Pynq-Z2"
fpga_part = pynq_part_map[pynq_board]
target_clk_ns = 10  # 100 MHz. This is the standard clock frequency used in the FINN tutorial.

dict_keys(['Ultra96', 'Ultra96-V2', 'Pynq-Z1', 'Pynq-Z2', 'ZCU102', 'ZCU104', 'ZCU111', 'RFSoC2x2', 'RFSoC4x2', 'KV260_SOM'])


In [52]:
from finn.transformation.fpgadataflow.specialize_layers import SpecializeLayers
model = ModelWrapper(dataflow_model_filename)

# Set all preferred_impl_style attributes to "rtl".
for node in model.graph.node:
    node_inst = getCustomOp(node)
    node_inst.set_nodeattr("preferred_impl_style", "rtl")

model = model.transform(SpecializeLayers(fpga_part))

# Show the source code of the SpecializeLayers transformation
#showSrc(SpecializeLayers)

# Save the FPGA HLS model
fpga_hls_path = f'./09_fpga_hls/{onnx_name}_fpga_hls.onnx'
Path(fpga_hls_path).parent.mkdir(parents=True, exist_ok=True)
model.save(fpga_hls_path)

/home/jose-vitor/finn-repo/src/finn/transformation/fpgadataflow/specialize_layers.py:155: UserWarning: There is no RTL variant for VVAU_. The node will automatically be
                        set to HLS variant. Please check the bit-widths to be <= 8 and ensure the
                        thresholds are implemented as standalone layer. Note that the RTL-variant
                        of this layer is only supported on Versal boards
  warnings.warn(warn_str)
/home/jose-vitor/finn-repo/src/finn/transformation/fpgadataflow/specialize_layers.py:166: UserWarning: There is no RTL variant of DuplicateStreams. Node DuplicateStreams_Thresholding_MultiThreshold_5 will automatically be
                        set to HLS variant.
  warnings.warn(warn_str)
/home/jose-vitor/finn-repo/src/finn/transformation/fpgadataflow/specialize_layers.py:166: UserWarning: There is no RTL variant of DuplicateStreams. Node DuplicateStreams_Thresholding_MultiThreshold_9 will automatically be
                      

In [54]:
showInNetron(fpga_hls_path)

Serving './09_fpga_hls/quant_model_8_bits_fpga_hls.onnx' at http://0.0.0.0:8081


### FOLDING MODEL

Folding in FINN describes how much a layer is time-multiplexed in terms of execution resources. There are several folding factors for each layer, controlled by the PE (parallelization over outputs) and SIMD (parallelization over inputs) parameters as described by the original [FINN paper](https://arxiv.org/pdf/1612.07119). The higher the PE and SIMD values are set, the faster the generated accelerator will run, and the more FPGA resources it will consume.

- PE = Processing Elements => aka actual computing units that will run MAC operations.
    - Theorically improves computing throughput.

- SIMD = Single Instruction Multiple Data => aka a way to feed MULTIPLE data at ONCE in a signle PE
    - Theorically improves datapath bandwidth.

One can set folding parameters manually by following the steps of the FINN example notebook [tfc_end2end_example.ipynb](https://github.com/Xilinx/finn/blob/main/notebooks/end2end_example/bnn-pynq/tfc_end2end_example.ipynb). 

This project is using the SetFolding transform to define automatically the PE ans SIMD parameter depending on target requirements. This is being done because it worked very well for the optimized Fast-SCNN model.

If your project requires more tinkering regarding folding, then I will leave the research for you.

In [23]:
from finn.transformation.fpgadataflow.set_folding import SetFolding

TARGET_FPS = 15
target_clk_cycles_per_frame = 1/(TARGET_FPS * 10e-9)

model = ModelWrapper(fpga_hls_path)
# Actual method SetFolding that will automatically modify the folding settings
model.transform(SetFolding(target_cycles_per_frame=target_clk_cycles_per_frame, mvau_wwidth_max=64, two_pass_relaxation=True))

# Blocking the URAM usage since Pynq-Z2 does not have URAMs
for n in model.graph.node:
    try:
        inst = getCustomOp(n)
        # If the hw block supports RAN config, then we force BRAM usage
        if "ram_style" in inst.get_nodeattr_types():
            inst.set_nodeattr("ram_style", "block")
    except:
        # If a node is not a custom op, we just skip it
        continue

# Save the FPGA HLS model
folded_model_path = f'./10_folded_model/{onnx_name}_folded_model.onnx'
Path(folded_model_path).parent.mkdir(parents=True, exist_ok=True)
model.save(folded_model_path)

/home/jose-vitor/finn-repo/src/finn/transformation/fpgadataflow/set_folding.py:221: UserWarning: SetFolding doesn't know how to handle op_type FMPadding_rtl
  warnings.warn("SetFolding doesn't know how to handle op_type " + op_type)
/home/jose-vitor/finn-repo/src/finn/transformation/fpgadataflow/set_folding.py:221: UserWarning: SetFolding doesn't know how to handle op_type UpsampleNearestNeighbour_hls
  warnings.warn("SetFolding doesn't know how to handle op_type " + op_type)
/home/jose-vitor/finn-repo/src/finn/transformation/fpgadataflow/set_folding.py:221: UserWarning: SetFolding doesn't know how to handle op_type StreamingConcat_hls
  warnings.warn("SetFolding doesn't know how to handle op_type " + op_type)


In [24]:
model = ModelWrapper(folded_model_path)
print(helper.printable_graph(model.graph))

graph main_graph (
  %R5U8yj[FLOAT, 1x1024x1024x3]
) initializers (
  %MultiThreshold_0_param0[FLOAT, 3x255]
  %MultiThreshold_1_param0[FLOAT, 32x255]
  %MultiThreshold_2_param0[FLOAT, 32x255]
  %MultiThreshold_3_param0[FLOAT, 48x255]
  %MultiThreshold_4_param0[FLOAT, 48x255]
  %MultiThreshold_5_param0[FLOAT, 64x255]
  %MultiThreshold_8_param0[FLOAT, 384x255]
  %MultiThreshold_9_param0[FLOAT, 64x255]
  %MultiThreshold_12_param0[FLOAT, 384x255]
  %MultiThreshold_13_param0[FLOAT, 64x255]
  %MultiThreshold_14_param0[FLOAT, 64x255]
  %MultiThreshold_18_param0[FLOAT, 384x255]
  %MultiThreshold_19_param0[FLOAT, 64x255]
  %MultiThreshold_20_param0[FLOAT, 64x255]
  %MultiThreshold_22_param0[FLOAT, 384x255]
  %MultiThreshold_23_param0[FLOAT, 384x255]
  %MultiThreshold_24_param0[FLOAT, 96x255]
  %MultiThreshold_27_param0[FLOAT, 576x255]
  %MultiThreshold_28_param0[FLOAT, 96x255]
  %MultiThreshold_29_param0[FLOAT, 96x255]
  %MultiThreshold_33_param0[FLOAT, 576x255]
  %MultiThreshold_34_param0[FLO

In [ ]:
showInNetron(folded_model_path)

## HARDWARE BUILD

### First, run an estimate

We first run some estimations based on the TARGET FPS constant defined earlier.

In [26]:
# ESTIMATE
import finn.builder.build_dataflow as build
import finn.builder.build_dataflow_config as build_cfg
import os
import shutil

model_file = folded_model_path

estimates_output_dir = "./output_estimates_only"

#Delete previous run results if exist
if os.path.exists(estimates_output_dir):
    shutil.rmtree(estimates_output_dir)
    print("Previous run results deleted!")


cfg_estimates = build.DataflowBuildConfig(
    output_dir          = estimates_output_dir,
    mvau_wwidth_max     = 80,
    target_fps          = TARGET_FPS,
    synth_clk_period_ns = 10.0,
    fpga_part           = "xc7z020clg400-1",
    steps               = build_cfg.estimate_only_dataflow_steps,
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
    ]
)

build.build_dataflow_cfg(model_file, cfg_estimates)

Previous run results deleted!
Building dataflow accelerator from ./10_folded_model/quant_model_8_bits_folded_model.onnx
Intermediate outputs will be generated in /tmp/finn_dev_jose-vitor
Final outputs will be generated in ./output_estimates_only
Build log is at ./output_estimates_only/build_dataflow.log
Running step: step_qonnx_to_finn [1/10]
Running step: step_tidy_up [2/10]
Running step: step_streamline [3/10]
Running step: step_convert_to_hw [4/10]
Running step: step_create_dataflow_partition [5/10]
Running step: step_specialize_layers [6/10]
Running step: step_target_fps_parallelization [7/10]
Running step: step_apply_folding_config [8/10]
Running step: step_minimize_bit_width [9/10]
Running step: step_generate_estimate_reports [10/10]
Completed successfully


0

### Read the estimates reports

In these reports, we can see the estimates fits our target, and we can get an overview of hw many LUTs will be used

In [13]:
estimates_output_dir = "./output_estimates_only"

In [27]:
! cat {estimates_output_dir}/report/estimate_network_performance.json

{
  "critical_path_cycles": 343958545,
  "max_cycles": 6291456,
  "max_cycles_node_name": "MVAU_rtl_1",
  "estimated_throughput_fps": 15.894571940104166,
  "estimated_latency_ns": 3439585450.0
}

In [28]:
from my_finn_utils import read_json_dict, pynqz2_viability_check

resource_dict = read_json_dict(f"{estimates_output_dir}/report/estimate_layer_resources.json")
pynqz2_viability_check(resource_dict)

Resource usage totals: {'BRAM_18K': 1264.0, 'LUT': 1013880.0, 'URAM': 9.0, 'DSP': 313.0}

Checking LUT: 1013880.0 used, 53200.0 limit. Usage percentage: 1905.79%. FAIL
Checking BRAM_18K: 1264.0 used, 280.0 limit. Usage percentage: 451.43%. FAIL
Checking DSP: 313.0 used, 220.0 limit. Usage percentage: 142.27%. FAIL
Checking URAM: 9.0 used, 0.0 limit. Usage percentage: 0.00%. IMPOSSIBLE (resource not available on Pynq-Z2)



In [29]:
! cat {estimates_output_dir}/report/estimate_layer_resources.json
#! cat {estimates_output_dir}/report/estimate_layer_cycles.json

{
  "Thresholding_rtl_0": {
    "BRAM_18K": 0,
    "BRAM_efficiency": 1,
    "LUT": 144.0,
    "URAM": 0,
    "URAM_efficiency": 1,
    "DSP": 0
  },
  "ConvolutionInputGenerator_rtl_0": {
    "BRAM_18K": 6,
    "BRAM_efficiency": 1,
    "LUT": 300,
    "URAM": 0,
    "URAM_efficiency": 1,
    "DSP": 0
  },
  "MVAU_rtl_0": {
    "BRAM_18K": 8,
    "BRAM_efficiency": 0.046875,
    "LUT": 0,
    "URAM": 0,
    "URAM_efficiency": 1,
    "DSP": 9
  },
  "Thresholding_rtl_1": {
    "BRAM_18K": 0,
    "BRAM_efficiency": 1,
    "LUT": 2860.0,
    "URAM": 0,
    "URAM_efficiency": 1,
    "DSP": 0
  },
  "FMPadding_rtl_0": {
    "BRAM_18K": 0,
    "BRAM_efficiency": 1,
    "LUT": 0,
    "URAM": 0,
    "URAM_efficiency": 1,
    "DSP": 0
  },
  "ConvolutionInputGenerator_rtl_1": {
    "BRAM_18K": 32,
    "BRAM_efficiency": 1,
    "LUT": 300,
    "URAM": 0,
    "URAM_efficiency": 1,
    "DSP": 0
  },
  "VVAU_hls_0": {
    "BRAM_18K": 1,
    "BRAM_efficiency": 0.125,
    "LUT": 797,
    "URAM": 0,


### Run the actual harware build


This step can take a lot of time.

Make sure you did your dev enrionement setup right, aka all the ENV varibles that should point on the xilinx tools are sourced

In [3]:
# Actual hardware build, ZYNQ BUILD
from finn.transformation.fpgadataflow.make_zynq_proj import ZynqBuild
from pathlib import Path

model = ModelWrapper(folded_model_path)
model = model.transform(ZynqBuild(platform = pynq_board, period_ns = target_clk_ns,partition_model_dir="./test",enable_debug=True))

# Save the FPGA post synthesis model
post_synth_path = f'./11_post_synth/{onnx_name}_post_synth.onnx'
Path(post_synth_path).parent.mkdir(parents=True, exist_ok=True)
model.save(post_synth_path)

/home/jose-vitor/finn-repo/src/finn/transformation/fpgadataflow/floorplan.py:107: UserWarning: 228 nodes have no entry in the provided floorplan, SLR was set to -1
  warnings.warn(


ValueError: cannot reshape array of size 32768 into shape (1,32,32,0,128)

In [ ]:
showInNetron(post_synth_path)